# 1. Security Models — Shared Responsibility, Defense in Depth, Zero Trust

These three models are the **mental frameworks** behind every security decision in the cloud. The exam tests whether you understand *which model applies when* and *what each one means practically*.

## Setup

```bash
cd security/sc-900/01-security-concepts
uv sync
```

Select the `SC-900 (Python)` kernel (top-right). If it's missing, `Cmd+Shift+P` → **Reload Window**.

---
## 1. The Shared Responsibility Model

**Key idea**: when you move to the cloud, security responsibility is *split* between you and the cloud provider. What you're responsible for changes depending on the service model.

| Layer | On-Premises | IaaS | PaaS | SaaS |
|-------|:-----------:|:----:|:----:|:----:|
| **Data & access** | You | You | You | You |
| **Applications** | You | You | You | Provider |
| **Identity & directory** | You | You | Shared | Provider |
| **Network controls** | You | You | Shared | Provider |
| **Operating system** | You | You | Provider | Provider |
| **Physical host** | You | Provider | Provider | Provider |
| **Physical network** | You | Provider | Provider | Provider |
| **Physical datacenter** | You | Provider | Provider | Provider |

**The rule that never changes**: *You are always responsible for your data, accounts, and access management.* Even in SaaS (like Microsoft 365), if you give everyone admin access, that's on you.

### Why it matters for the exam

Questions ask things like: *"In a PaaS deployment, who is responsible for patching the operating system?"* → **The cloud provider**.

Let's make this interactive:

In [ ]:
RESPONSIBILITIES = {
    'Physical datacenter': {'on-prem': 'customer', 'iaas': 'provider', 'paas': 'provider', 'saas': 'provider'},
    'Physical network':    {'on-prem': 'customer', 'iaas': 'provider', 'paas': 'provider', 'saas': 'provider'},
    'Physical host':       {'on-prem': 'customer', 'iaas': 'provider', 'paas': 'provider', 'saas': 'provider'},
    'Operating system':    {'on-prem': 'customer', 'iaas': 'customer', 'paas': 'provider', 'saas': 'provider'},
    'Network controls':    {'on-prem': 'customer', 'iaas': 'customer', 'paas': 'shared',   'saas': 'provider'},
    'Identity':            {'on-prem': 'customer', 'iaas': 'customer', 'paas': 'shared',   'saas': 'provider'},
    'Applications':        {'on-prem': 'customer', 'iaas': 'customer', 'paas': 'customer', 'saas': 'provider'},
    'Data & access':       {'on-prem': 'customer', 'iaas': 'customer', 'paas': 'customer', 'saas': 'customer'},
}

def quiz_shared_responsibility():
    import random
    layers = list(RESPONSIBILITIES.keys())
    models = ['iaas', 'paas', 'saas']
    score = 0
    questions = 5
    for _ in range(questions):
        layer = random.choice(layers)
        model = random.choice(models)
        correct = RESPONSIBILITIES[layer][model]
        answer = input(f'In {model.upper()}, who handles "{layer}"? (customer/provider/shared): ').strip().lower()
        if answer == correct:
            print(f'  ✅ Correct!')
            score += 1
        else:
            print(f'  ❌ Wrong — the answer is: {correct}')
    print(f'\nScore: {score}/{questions}')

quiz_shared_responsibility()

---
## 2. Defense in Depth

**Key idea**: security is layered like an onion. If one layer fails, the next one catches the threat. No single mechanism is enough.

The layers (from outside to inside):

```
┌─────────────────────────────────────────┐
│           Physical Security             │  Locked datacenters, badge access
│  ┌───────────────────────────────────┐  │
│  │      Identity & Access            │  │  Entra ID, MFA, Conditional Access
│  │  ┌─────────────────────────────┐  │  │
│  │  │       Perimeter             │  │  │  DDoS protection, firewalls
│  │  │  ┌───────────────────────┐  │  │  │
│  │  │  │     Network           │  │  │  │  NSGs, subnets, private endpoints
│  │  │  │  ┌─────────────────┐  │  │  │  │
│  │  │  │  │   Compute       │  │  │  │  │  VM hardening, patching
│  │  │  │  │  ┌───────────┐  │  │  │  │  │
│  │  │  │  │  │Application│  │  │  │  │  │  Secure code, input validation
│  │  │  │  │  │ ┌───────┐ │  │  │  │  │  │
│  │  │  │  │  │ │ Data  │ │  │  │  │  │  │  Encryption at rest & in transit
│  │  │  │  │  │ └───────┘ │  │  │  │  │  │
│  │  │  │  │  └───────────┘  │  │  │  │  │
│  │  │  │  └─────────────────┘  │  │  │  │
│  │  │  └───────────────────────┘  │  │  │
│  │  └─────────────────────────────┘  │  │
│  └───────────────────────────────────┘  │
└─────────────────────────────────────────┘
```

### What fails if we only have one layer?

In [ ]:
LAYERS = [
    ('Physical',    'Locked doors, cameras, biometrics'),
    ('Identity',    'Entra ID, MFA, Conditional Access'),
    ('Perimeter',   'Azure Firewall, DDoS Protection, WAF'),
    ('Network',     'NSGs, subnets, private endpoints, VPN'),
    ('Compute',     'OS patching, endpoint protection, Bastion'),
    ('Application', 'Secure coding, input validation, secrets management'),
    ('Data',        'Encryption at rest, in transit, access control'),
]

ATTACKS = [
    {'name': 'Phishing email → stolen password',
     'blocked_by': ['Identity'],
     'explanation': 'MFA blocks the attacker even with the password'},
    {'name': 'DDoS attack flooding the public IP',
     'blocked_by': ['Perimeter'],
     'explanation': 'Azure DDoS Protection absorbs the traffic'},
    {'name': 'Attacker on the same VNet scans ports',
     'blocked_by': ['Network'],
     'explanation': 'NSG rules deny traffic from unauthorized subnets'},
    {'name': 'SQL injection in a web form',
     'blocked_by': ['Application', 'Perimeter'],
     'explanation': 'Input validation + WAF both catch it'},
    {'name': 'Stolen database backup',
     'blocked_by': ['Data'],
     'explanation': 'Encryption at rest makes the backup useless without keys'},
    {'name': 'Unpatched VM exploit',
     'blocked_by': ['Compute'],
     'explanation': 'Regular patching + Defender for Endpoint detects exploitation'},
]

print('Defense in Depth: How layers stop attacks\n')
for attack in ATTACKS:
    blockers = ', '.join(attack['blocked_by'])
    print(f'🔴 Attack: {attack["name"]}')
    print(f'   🛡️  Blocked by: {blockers} layer')
    print(f'   💡 Why: {attack["explanation"]}\n')

### Exam tip

Defense in depth is about **multiple layers catching different threats**. If asked *"which model uses a layered approach to security?"* → **Defense in depth** (not Zero Trust, not shared responsibility).

---
## 3. Zero Trust

**Key idea**: "never trust, always verify". Unlike the old castle-and-moat model (trust everyone inside the network), Zero Trust assumes breach and verifies every request.

### The three principles

1. **Verify explicitly** — always authenticate and authorize based on all available data (identity, location, device, service, data classification, anomalies).
2. **Least privilege access** — limit user access with Just-In-Time and Just-Enough-Access (JIT/JEA).
3. **Assume breach** — minimize blast radius using segmentation, end-to-end encryption, analytics.

### The six pillars

Zero Trust applies across six foundational elements:

| Pillar | What it means | Azure example |
|--------|--------------|---------------|
| **Identity** | Verify every user and service | Entra ID + MFA + Conditional Access |
| **Devices** | Ensure devices meet health requirements | Intune compliance policies |
| **Applications** | Manage shadow IT and in-app permissions | Defender for Cloud Apps |
| **Data** | Classify, label, and encrypt | Purview sensitivity labels |
| **Infrastructure** | Detect attacks and block risky behavior | Defender for Cloud |
| **Network** | Segment, encrypt in transit, real-time threat protection | NSGs, Private Link, Azure Firewall |

Let's simulate what Zero Trust looks like in code:

In [ ]:
import json
from dataclasses import dataclass

@dataclass
class AccessRequest:
    user: str
    role: str
    mfa_verified: bool
    device_compliant: bool
    location: str
    resource: str

POLICIES = [
    {'name': 'Require MFA',
     'check': lambda r: r.mfa_verified,
     'deny_reason': 'MFA not completed'},
    {'name': 'Device compliance',
     'check': lambda r: r.device_compliant,
     'deny_reason': 'Device is not compliant (no encryption, outdated OS, etc.)'},
    {'name': 'Block unknown locations',
     'check': lambda r: r.location in ('office', 'home-vpn', 'trusted-country'),
     'deny_reason': f'Access from untrusted location'},
    {'name': 'Least privilege (role check)',
     'check': lambda r: r.resource != 'admin-portal' or r.role == 'admin',
     'deny_reason': 'Insufficient role for admin portal'},
]

def evaluate_zero_trust(request: AccessRequest) -> dict:
    """Simulate a Conditional Access policy engine."""
    results = []
    allowed = True
    for policy in POLICIES:
        passed = policy['check'](request)
        results.append({
            'policy': policy['name'],
            'result': '✅ PASS' if passed else '❌ DENY',
            'reason': None if passed else policy['deny_reason'],
        })
        if not passed:
            allowed = False
    return {'allowed': allowed, 'checks': results}

# --- Scenario 1: Good request ---
print('=== Scenario 1: Employee from office, MFA done, compliant device ===')
r1 = AccessRequest('alice@contoso.com', 'reader', True, True, 'office', 'files')
print(json.dumps(evaluate_zero_trust(r1), indent=2))

# --- Scenario 2: No MFA ---
print('\n=== Scenario 2: Same employee, forgot MFA ===')
r2 = AccessRequest('alice@contoso.com', 'reader', False, True, 'office', 'files')
print(json.dumps(evaluate_zero_trust(r2), indent=2))

# --- Scenario 3: Unknown location ---
print('\n=== Scenario 3: Logging in from an unknown country ===')
r3 = AccessRequest('alice@contoso.com', 'reader', True, True, 'unknown-country', 'files')
print(json.dumps(evaluate_zero_trust(r3), indent=2))

# --- Scenario 4: Non-admin trying admin portal ---
print('\n=== Scenario 4: Reader tries to access admin portal ===')
r4 = AccessRequest('alice@contoso.com', 'reader', True, True, 'office', 'admin-portal')
print(json.dumps(evaluate_zero_trust(r4), indent=2))

### What just happened?

This is exactly what **Conditional Access** does in Microsoft Entra ID:

1. Every request is evaluated against multiple policies (verify explicitly).
2. Even a valid identity gets blocked if any condition fails (assume breach).
3. Alice only gets access to what her role permits (least privilege).

In the real Azure world, these policies are configured in the Entra portal or via Azure CLI — we'll cover that in Lab 2.

### Exam tip

Zero Trust has **three principles** (verify explicitly, least privilege, assume breach) and **six pillars** (identity, devices, applications, data, infrastructure, network). The exam loves asking you to match scenarios to principles.

---
## 4. Governance, Risk, and Compliance (GRC)

This section is mostly conceptual, but it's critical for the exam.

### The three pieces

| | What it is | Example |
|-|-----------|----------|
| **Governance** | Rules and policies an organization enforces | "All data must be encrypted at rest" |
| **Risk** | Identifying and managing threats | "If we don't encrypt, a breach costs us $10M" |
| **Compliance** | Proving you follow external regulations | GDPR, HIPAA, SOC 2, ISO 27001 |

### Microsoft tools for GRC

| Tool | What it does |
|------|--------------|
| **Microsoft Purview Compliance Manager** | Tracks your compliance posture with a compliance *score*. Suggests improvement actions. |
| **Azure Policy** | Enforces rules on Azure resources (e.g., "all storage accounts must use HTTPS") |
| **Service Trust Portal** | Where Microsoft publishes audit reports, compliance docs, pen test results |
| **Microsoft Priva** | Manages privacy risk in Microsoft 365 data |

### Data residency & data sovereignty

- **Data residency**: *where* your data is physically stored. You choose the Azure region.
- **Data sovereignty**: the data is subject to the *laws of the country* where it's stored. EU data stored in Germany falls under German + EU law.

The exam may ask: *"What ensures that data is stored in a specific geographic location?"* → **Data residency**.

In [ ]:
# Quick quiz: GRC concepts
quiz = [
    ('An organization requires all VMs to have disk encryption enabled. What GRC concept is this?',
     'governance', {'governance': 'It\'s a rule the org enforces internally'}),
    ('A hospital must follow HIPAA when handling patient data. What GRC concept?',
     'compliance', {'compliance': 'HIPAA is an external regulation'}),
    ('The security team estimates a data breach would cost $5M and decides to invest in DLP. What GRC concept?',
     'risk', {'risk': 'They\'re identifying and mitigating a threat based on potential impact'}),
    ('EU customer data must stay in the EU. What is this called?',
     'data residency', {'data residency': 'It\'s about the physical location of storage'}),
]

score = 0
for q, correct, explanations in quiz:
    ans = input(f'Q: {q}\n> ').strip().lower()
    if ans == correct:
        print(f'✅ Correct! {explanations[correct]}\n')
        score += 1
    else:
        print(f'❌ The answer is: {correct}. {explanations[correct]}\n')
print(f'Score: {score}/{len(quiz)}')

---
## Summary — what to remember for the exam

| Model | Core idea | Key phrase |
|-------|-----------|------------|
| Shared responsibility | Security duties are split between you and Microsoft | "You always own your data and identities" |
| Defense in depth | Layered security from physical to data | "If one layer fails, the next catches it" |
| Zero Trust | Never trust, always verify | "Verify explicitly, least privilege, assume breach" |
| GRC | Governance = internal rules, Risk = threats, Compliance = external regulations | "Compliance Manager gives you a score" |

**Next**: [Notebook 2 — Encryption and Hashing](02_encryption_and_hashing.ipynb)